# Lab 21 — Dimensionality Reduction: PCA, t-SNE, and UMAP

This lab covers the three dimensionality-reduction methods you explored this week: PCA (linear, variance-maximizing), t-SNE (non-linear, local-structure-preserving), and UMAP (non-linear, faster, better global structure). You'll fit each one, interpret what they actually show, and practice the "start simple, escalate only when needed" workflow.

**Concepts covered:** principal components and variance explained, PCA loadings/biplots, t-SNE perplexity and its interpretation limits, UMAP's n_neighbors/min_dist, and choosing between methods.

**Reference working sessions:**
- working-sessions/unsupervised/07_pca.ipynb
- working-sessions/unsupervised/08_tsne.ipynb
- working-sessions/unsupervised/09_umap.ipynb
- working-sessions/unsupervised/10_dimensionality_reduction_comparison.ipynb

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_california_housing,
)

# Answer key — encoded to prevent students from reading answers directly
_ak = make_answer_key({
    'q1': 'B',
    'q2': 'C',
    'q3': 'B',
    'q4': 'B',
})

## Section A — Multiple Choice

Answer each question below by setting the variable to `A`, `B`, `C`, or `D`.

In [ ]:
# Q1 — Per working-sessions/unsupervised/07_pca.ipynb, what does the first
# principal component (PC1) represent?
#
#   A) The single original feature most correlated with the target variable
#   B) The direction in the (scaled) feature space along which the data has
#      the most variance
#   C) A weighted average of all class labels
#   D) The feature with the smallest range in the original units

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", \
    "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — revisit working-sessions/unsupervised/07_pca.ipynb and its " \
    "\"How it learns\" section."
print("✓ Question 1 correct!")

In [ ]:
# Q2 — Per working-sessions/unsupervised/08_tsne.ipynb, which of the
# following is a VALID conclusion to draw from a t-SNE scatter plot?
#
#   A) Two tight, well-separated clusters are far apart in the original
#      high-dimensional space
#   B) A cluster that looks twice as large as another contains roughly twice
#      the density of points
#   C) Points plotted close together were likely close together (similar) in
#      the original high-dimensional space
#   D) The same t-SNE settings run twice will always produce an identical
#      layout

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", \
    "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — revisit working-sessions/unsupervised/08_tsne.ipynb and its " \
    "\"What's happening?\" critical interpretation rules."
print("✓ Question 2 correct!")

In [ ]:
# Q3 — Per working-sessions/unsupervised/09_umap.ipynb, what is one
# practical advantage UMAP has over t-SNE?
#
#   A) UMAP is guaranteed to always run faster, regardless of sample size
#   B) UMAP can project new, previously unseen data points without rerunning
#      the whole algorithm
#   C) UMAP requires no hyperparameters at all
#   D) UMAP is deterministic even without setting a random_state

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", \
    "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — revisit working-sessions/unsupervised/09_umap.ipynb and its " \
    "\"Strengths and weaknesses\" table."
print("✓ Question 3 correct!")

In [ ]:
# Q4 — Per working-sessions/unsupervised/10_dimensionality_reduction_comparison.ipynb's
# rule of thumb, what's the recommended order to try dimensionality-reduction
# methods in?
#
#   A) Always start with t-SNE since it gives the most detailed local view
#   B) Start with PCA; if it doesn't show clear separation, try UMAP; reach
#      for t-SNE only if UMAP misses important local structure
#   C) Randomly try all three and pick whichever finishes fastest
#   D) Start with UMAP always, since it's the newest method

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", \
    "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — revisit working-sessions/unsupervised/10_dimensionality_reduction_comparison.ipynb " \
    "and its \"Key hyperparameters\" rule of thumb."
print("✓ Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: What PC1 represents"),
    (q2_answer, _ak['q2'], "Q2: Valid t-SNE interpretation"),
    (q3_answer, _ak['q3'], "Q3: UMAP's advantage over t-SNE"),
    (q4_answer, _ak['q4'], "Q4: Recommended method order"),
], total=4)

## Section B — Coding Exercises

Fill in each `___` blank. All three exercises use the California housing dataset you already scaled in `working-sessions/unsupervised/07_pca.ipynb`.

### B1 — Fit PCA and Measure Variance Captured

Scale the California housing features and fit a 2-component PCA. Report how much of the total variance those 2 components capture.

In [ ]:
X_b1, y_b1 = load_california_housing()

scaler_b1 = ___()   # YOUR CODE — the preprocessing step that puts every feature on a comparable scale before PCA
X_b1_scaled = scaler_b1.___(X_b1)   # YOUR CODE — fit the scaler and transform the features in one call

pca_b1 = ___(n_components=2, random_state=42)   # YOUR CODE — the dimensionality-reduction model introduced in 07_pca.ipynb
Z_b1 = pca_b1.___(X_b1_scaled)   # YOUR CODE — fit the model and produce the 2D projection in one call

var_explained_b1 = pca_b1.explained_variance_ratio_.sum()

assert Z_b1.shape == (len(X_b1_scaled), 2), \
    "Projection should have one row per district and 2 columns"
assert 0.3 < var_explained_b1 < 0.7, \
    "With 2 of 8 components you should capture a meaningful but partial share of the variance"
print(f"✓ B1 complete! 2 components capture {var_explained_b1:.1%} of the variance.")

### B2 — Fit t-SNE on a Sample

t-SNE is too slow to run on all 20,640 districts in a lab exercise, so work on an 800-row random sample instead. Fit t-SNE with `perplexity=30` and confirm the optimizer actually ran.

In [ ]:
rng_b2 = np.random.RandomState(42)
sample_idx_b2 = rng_b2.choice(len(X_b1_scaled), size=800, replace=False)
X_b2_sample = X_b1_scaled[sample_idx_b2]

tsne_b2 = ___(n_components=2, perplexity=30, random_state=42, init='pca')   # YOUR CODE — the non-linear method introduced in 08_tsne.ipynb
Z_b2 = tsne_b2.___(X_b2_sample)   # YOUR CODE — fit the model and produce the 2D projection in one call

assert Z_b2.shape == (800, 2), \
    "Projection should have one row per sampled point and 2 columns"
assert tsne_b2.n_iter_ > 0, \
    "The optimizer should have run at least one iteration"
print(f"✓ B2 complete! t-SNE ran for {tsne_b2.n_iter_} iterations.")

### B3 — Fit UMAP

Fit UMAP with the same `n_neighbors` and `min_dist` defaults used in `working-sessions/unsupervised/09_umap.ipynb`, and confirm the projection has the expected shape.

In [ ]:
if UMAP_AVAILABLE:
    reducer_b3 = ___(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)   # YOUR CODE — the graph-based method introduced in 09_umap.ipynb
    Z_b3 = reducer_b3.___(X_b1_scaled)   # YOUR CODE — fit the model and produce the 2D projection in one call

    assert Z_b3.shape == (len(X_b1_scaled), 2), \
        "Projection should have one row per district and 2 columns"
    print(f"✓ B3 complete! UMAP projected {len(X_b1_scaled):,} districts to 2D.")
else:
    print("umap-learn is not installed — install it with `pip install umap-learn` and rerun this cell.")

## Section C — Applied Problem

You've been asked to visualize the California housing dataset in 2D so a colleague can spot any obvious structure in home prices. Following the recommended workflow from `working-sessions/unsupervised/10_dimensionality_reduction_comparison.ipynb` ("start with PCA... escalate only when the linear view is insufficient"), you'll try PCA first, measure how well it separates price tiers using the silhouette metric from `06_clustering_evaluation.ipynb`, and then escalate to t-SNE.

In [ ]:
# --- Step 1: Load and prepare data ---
X_c, y_c = load_california_housing()

scaler_c = ___()   # YOUR CODE — the same scaling step used throughout this lab
X_c_scaled = scaler_c.fit_transform(X_c)

price_tier_c = pd.qcut(y_c, q=3, labels=['Low', 'Mid', 'High'])
tier_codes_c = price_tier_c.cat.codes.to_numpy()

# --- Step 2: Try PCA first (the recommended starting point) ---
pca_c = PCA(n_components=2, random_state=42)
Z_pca_c = pca_c.fit_transform(X_c_scaled)

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(
    x=Z_pca_c[:, 0], y=Z_pca_c[:, 1], hue=price_tier_c,
    palette=[PALETTE["primary"], PALETTE["accent"], PALETTE["secondary"]],
    alpha=0.4, s=15, ax=ax,
)
ax.set_title("PCA Projection Colored by Price Tier")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()

# --- Step 3: Quantify the separation you just saw ---
sil_pca_c = ___(Z_pca_c, tier_codes_c)   # YOUR CODE — the clustering-quality metric from working-sessions/unsupervised/06_clustering_evaluation.ipynb, applied here to the price tiers instead of cluster assignments

# --- Step 4: Escalate to t-SNE on a sample ---
rng_c = np.random.RandomState(42)
sample_idx_c = rng_c.choice(len(X_c_scaled), size=1000, replace=False)
X_c_sample = X_c_scaled[sample_idx_c]
tier_sample_codes_c = tier_codes_c[sample_idx_c]
tier_sample_c = pd.Categorical.from_codes(tier_sample_codes_c, categories=['Low', 'Mid', 'High'])

tsne_c = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
Z_tsne_c = tsne_c.___(X_c_sample)   # YOUR CODE — fit the model and produce the 2D projection in one call

sil_tsne_c = silhouette_score(Z_tsne_c, tier_sample_codes_c)

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(
    x=Z_tsne_c[:, 0], y=Z_tsne_c[:, 1], hue=tier_sample_c,
    palette=[PALETTE["primary"], PALETTE["accent"], PALETTE["secondary"]],
    alpha=0.5, s=20, ax=ax,
)
ax.set_title("t-SNE Projection Colored by Price Tier (1,000-row sample)")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
plt.show()

assert -1 <= sil_pca_c <= 1, "Silhouette score must be between -1 and 1"
assert -1 <= sil_tsne_c <= 1, "Silhouette score must be between -1 and 1"
print(f"✓ Section C complete! PCA silhouette: {sil_pca_c:.3f} | t-SNE silhouette: {sil_tsne_c:.3f}")

## Section D — Reflection

These questions are for reflection. Edit the markdown cells below each question with your own answer — there are no wrong answers here. Your instructor may review these if requested.

**Question D1**

In `07_pca.ipynb`, the biplot widget showed that the price tiers do NOT separate well in PCA's first two components, even though PCA is working exactly as designed. Why might a dimensionality-reduction method fail to separate the outcome you actually care about, even when it's doing its job correctly?

*Your response here...*

**Question D2**

In `10_dimensionality_reduction_comparison.ipynb`'s speed-comparison widget, UMAP was not measured as faster than t-SNE at the sample sizes tested — even though the literature and this notebook's own reference tables describe UMAP as generally faster. What does this tell you about applying general algorithm benchmarks to your own specific dataset and environment?

*Your response here...*

**Question D3**

Look back at the "When to use each" guidance in `10_dimensionality_reduction_comparison.ipynb`. Pick a real or hypothetical dataset from your own experience or interests. Which method would you reach for first, and what would make you escalate to a different one?

*Your response here...*